# **1. Perkenalan Dataset**


**Sumber Dataset**: Kaggle — FIFA World Cup 2026 Player Performance oleh Rauffauzanrambe.

Dataset ini berisi statistik performa pemain selama Piala Dunia FIFA 2026, mencakup 54.600 baris data dengan 75 kolom yang meliputi informasi demografis pemain (usia, kebangsaan, posisi, klub), statistik pertandingan (menit bermain, gol, assist, operan, tembakan), dan metrik performa lanjutan (expected goals, rating pemain, kontribusi ofensif/defensif).

**Tujuan Proyek (Multi-Output):**
1. **Regresi** — Memprediksi jumlah gol (*goals*) yang akan dicetak pemain.
2. **Klasifikasi** — Mengklasifikasikan posisi pemain (*position*) ke dalam 4 kategori: Defender, Forward, Goalkeeper, Midfielder.

Pendekatan multi-output ini memungkinkan model untuk belajar representasi fitur yang menangkap kedua aspek performa secara simultan.

# **2. Import Library**

Pada tahap ini, dilakukan import pustaka Python yang dibutuhkan untuk analisis data, preprocessing, dan pembangunan model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

# **3. Memuat Dataset**

Dataset dimuat dari file CSV menggunakan pandas. Dilakukan pengecekan awal untuk memahami struktur data.

In [ ]:
DATA_PATH = 'fifa_world_cup_2026_player_performance.csv'
df = pd.read_csv(DATA_PATH)

print(f'Dataset shape: {df.shape}')
print(f'Number of rows: {df.shape[0]}')
print(f'Number of columns: {df.shape[1]}')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'\nData types:')
print(df.dtypes.value_counts())

# **4. Exploratory Data Analysis (EDA)**

EDA dilakukan untuk memahami karakteristik dataset, distribusi fitur, dan hubungan antar variabel.

### 4.1 Statistik Deskriptif

In [ ]:
df.describe()

### 4.2 Distribusi Target — Posisi Pemain (Klasifikasi)

In [ ]:
plt.figure(figsize=(8, 5))
position_counts = df['position'].value_counts()
colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']
bars = plt.bar(position_counts.index, position_counts.values, color=colors)
plt.title('Distribution of Player Positions', fontsize=14, fontweight='bold')
plt.xlabel('Position')
plt.ylabel('Count')
for bar, val in zip(bars, position_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             str(val), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nPosition distribution (percentage):')
print((position_counts / len(df) * 100).round(2).astype(str) + ' %')

### 4.3 Distribusi Target — Gol (Regresi)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

goals_dist = df['goals'].value_counts().sort_index()
axes[0].bar(goals_dist.index, goals_dist.values, color='#2196F3', edgecolor='black')
axes[0].set_title('Distribution of Goals Scored', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Goals')
axes[0].set_ylabel('Count')
for x, y in zip(goals_dist.index, goals_dist.values):
    axes[0].text(x, y + 200, str(y), ha='center', fontsize=9, fontweight='bold')

goals_by_pos = df.groupby('position')['goals'].mean().sort_values(ascending=False)
axes[1].bar(goals_by_pos.index, goals_by_pos.values, color=colors)
axes[1].set_title('Average Goals by Position', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Average Goals')
for i, v in enumerate(goals_by_pos.values):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

### 4.4 Distribusi Fitur Numerik Utama

In [ ]:
num_features = ['age', 'height_cm', 'weight_kg', 'market_value_eur',
                'minutes_played', 'pass_accuracy', 'player_rating',
                'stamina_score', 'distance_covered_km']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].hist(df[col], bins=40, color='#4CAF50', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### 4.5 Analisis Korelasi

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr_with_goals = numeric_df.corr()['goals'].sort_values(ascending=False)

plt.figure(figsize=(10, 8))
top_corrs = corr_with_goals.head(16)
colors_corr = ['#4CAF50' if v > 0 else '#F44336' for v in top_corrs.values]
plt.barh(range(len(top_corrs)), top_corrs.values, color=colors_corr)
plt.yticks(range(len(top_corrs)), top_corrs.index)
plt.title('Top 15 Features Correlated with Goals', fontsize=14, fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 12))
corr_matrix = numeric_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8},
            xticklabels=False, yticklabels=False)
plt.title('Correlation Matrix of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.6 Analisis Variabel Kategorikal

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

cat_cols = ['preferred_foot', 'tournament_stage', 'match_result']
for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[i].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
                startangle=90, colors=plt.cm.Set2(range(len(counts))))
    axes[i].set_title(f'{col} Distribution', fontweight='bold')

axes[3].axis('off')
axes[4].axis('off')
axes[5].axis('off')
plt.tight_layout()
plt.show()

### 4.7 Analisis Outlier

In [ ]:
outlier_features = ['market_value_eur', 'distance_covered_km',
                    'sprint_distance_km', 'top_speed_kmh',
                    'player_rating', 'performance_score']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(outlier_features):
    axes[i].boxplot(df[col], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#FF9800', alpha=0.7))
    axes[i].set_title(f'Boxplot of {col}', fontweight='bold')
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

Data preprocessing bertujuan membersihkan dan menyiapkan data agar siap digunakan untuk model machine learning.

### 5.1 Menghapus Kolom Identitas

In [ ]:
id_cols = ['player_id', 'player_name', 'match_id', 'match_date',
           'stadium', 'city', 'club_name', 'nationality', 'team', 'opponent_team']
df_processed = df.drop(columns=id_cols)
print(f'Shape after dropping ID columns: {df_processed.shape}')

### 5.2 Encoding Variabel Kategorikal

In [ ]:
position_mapping = {'Defender': 0, 'Forward': 1, 'Goalkeeper': 2, 'Midfielder': 3}
df_processed['position_encoded'] = df_processed['position'].map(position_mapping)
df_processed = df_processed.drop(columns=['position'])

stage_mapping = {
    'Group Stage': 0, 'Round of 32': 1, 'Round of 16': 2,
    'Quarter Finals': 3, 'Semi Finals': 4,
    'Third Place Match': 5, 'Final': 6
}
df_processed['tournament_stage'] = df_processed['tournament_stage'].map(stage_mapping)

result_mapping = {'L': 0, 'D': 1, 'W': 2}
df_processed['match_result'] = df_processed['match_result'].map(result_mapping)

foot_mapping = {'Left': 0, 'Right': 1}
df_processed['preferred_foot'] = df_processed['preferred_foot'].map(foot_mapping)

print(f'Shape after encoding: {df_processed.shape}')
print(f'Encoded columns: position_encoded, tournament_stage, match_result, preferred_foot')

### 5.3 Feature Scaling

In [ ]:
target_reg = 'goals'
target_cls = 'position_encoded'

feature_cols = [c for c in df_processed.columns
                if c not in [target_reg, target_cls]]

scaler = StandardScaler()
df_processed[feature_cols] = scaler.fit_transform(df_processed[feature_cols])

print(f'Features scaled using StandardScaler: {len(feature_cols)} features')
print(f'Final shape: {df_processed.shape}')

### 5.4 Train-Test Split

In [ ]:
X = df_processed.drop(columns=[target_reg, target_cls])
y_reg = df_processed[target_reg]
y_cls = df_processed[target_cls]

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_reg_train: {y_reg_train.shape}')
print(f'y_reg_test: {y_reg_test.shape}')
print(f'y_cls_train: {y_cls_train.shape}')
print(f'y_cls_test: {y_cls_test.shape}')

print(f'\nClass distribution in y_cls_train:')
print(y_cls_train.value_counts().sort_index())

### 5.5 Menyimpan Data yang Telah Diproses

In [ ]:
OUTPUT_DIR = 'Membangun_model/fifa_dataset_preprocessing'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(OUTPUT_DIR, 'fifa_data_clean.csv')
df_processed.to_csv(output_path, index=False)
print(f'Cleaned dataset saved to: {output_path}')
print(f'Final dataset shape: {df_processed.shape}')

### 5.6 Ringkasan Data Preprocessing

In [ ]:
print('=== Data Preprocessing Summary ===')
print(f'Original shape: {df.shape}')
print(f'Final shape: {df_processed.shape}')
print(f'\nDropped columns ({len(id_cols)}): {id_cols}')
print(f'Encoded columns: position, tournament_stage, match_result, preferred_foot')
print(f'Scaling method: StandardScaler (zero mean, unit variance)')
print(f'Number of features: {X.shape[1]}')
print(f'Regression target: goals')
print(f'Classification target: position_encoded ({len(position_mapping)} classes)')
print(f'Train-Test split: 80/20 with stratification on position_encoded')
print(f'Train size: {X_train.shape[0]} samples')
print(f'Test size: {X_test.shape[0]} samples')